# Contract Ingestion with GPU Acceleration

This notebook generates embeddings for your contracts using Colab's free GPU.

**Steps:**
1. Upload your contracts to Google Drive
2. Run this notebook
3. Download the generated embeddings file
4. Run the local loader script to insert into Qdrant

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q sentence-transformers PyMuPDF tqdm

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuration - UPDATE THIS PATH!
# Your contracts should be in: Google Drive/contracts/{year}/{quarter}/*.pdf or *.htm

DATA_DIR = "/content/drive/MyDrive/contracts"  # <-- Change this to your folder path
OUTPUT_FILE = "/content/drive/MyDrive/contract_embeddings.jsonl.gz"  # Compressed output
MODEL_NAME = "BAAI/bge-large-en-v1.5"  # Best quality (1024 dims)
# MODEL_NAME = "all-MiniLM-L6-v2"  # Faster/smaller (384 dims) - use if storage is tight
CHUNK_SIZE = 512
BATCH_SIZE = 64  # GPU can handle larger batches

# Storage estimates:
# - BGE-large: ~5-6GB uncompressed, ~2GB compressed
# - MiniLM: ~2GB uncompressed, ~0.8GB compressed
# Google Drive free tier: 15GB

In [ ]:
import json
import re
from pathlib import Path
from dataclasses import dataclass
from typing import List, Optional
from html.parser import HTMLParser
from io import StringIO

import fitz  # PyMuPDF
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import torch

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Section patterns for legal documents
SECTION_PATTERNS = [
    r'^ARTICLE\s+[IVXLCDM\d]+',
    r'^SECTION\s+\d+',
    r'^\d+\.\s+[A-Z][A-Z\s]{2,}',
    r'^[A-Z][A-Z\s]{15,}$',
    r'^(?:WHEREAS|RECITALS|DEFINITIONS|TERM|TERMINATION)',
    r'^(?:PAYMENT|CONFIDENTIALITY|INDEMNIFICATION|GOVERNING LAW)',
]

@dataclass
class Chunk:
    text: str
    section: str
    index: int

def estimate_tokens(text: str) -> int:
    return int(len(text.split()) * 1.3)

def is_section_header(line: str) -> bool:
    line = line.strip()
    if not line or len(line) < 3:
        return False
    return any(re.match(p, line, re.IGNORECASE) for p in SECTION_PATTERNS)

def chunk_text(text: str, max_tokens: int = 512) -> List[Chunk]:
    chunks = []
    current_section = "PREAMBLE"
    current_text = ""
    
    for line in text.split('\n'):
        if is_section_header(line):
            if current_text.strip():
                chunks.extend(_split_large_section(current_text, current_section, max_tokens))
            current_section = line.strip()[:100]
            current_text = ""
        else:
            current_text += line + "\n"
    
    if current_text.strip():
        chunks.extend(_split_large_section(current_text, current_section, max_tokens))
    
    return [Chunk(text=t.strip(), section=s, index=i) for i, (t, s) in enumerate(chunks)]

def _split_large_section(text: str, section: str, max_tokens: int) -> List[tuple]:
    if estimate_tokens(text) <= max_tokens:
        return [(f"{section}\n{text}" if section != "PREAMBLE" else text, section)]
    
    results = []
    paragraphs = text.split('\n\n')
    current = ""
    
    for para in paragraphs:
        test = current + "\n\n" + para if current else para
        if estimate_tokens(test) <= max_tokens:
            current = test
        else:
            if current:
                prefix = f"{section}\n" if not results else ""
                results.append((prefix + current, section))
            current = para
    
    if current:
        prefix = f"{section}\n" if not results else ""
        results.append((prefix + current, section))
    
    return results if results else [(text[:max_tokens * 4], section)]

In [ ]:
# Text extraction functions

def extract_text_from_pdf(pdf_path: Path) -> Optional[str]:
    try:
        doc = fitz.open(str(pdf_path))
        text = "\n\n".join(page.get_text() for page in doc)
        doc.close()
        return text if text.strip() else None
    except Exception as e:
        print(f"Failed to extract PDF {pdf_path.name}: {e}")
        return None

class HTMLTextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.text = StringIO()
        self.skip_tags = {'script', 'style', 'head', 'meta'}
        self.current_tag = None
    
    def handle_starttag(self, tag, attrs):
        self.current_tag = tag.lower()
    
    def handle_endtag(self, tag):
        if tag.lower() in {'p', 'div', 'br', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li', 'tr'}:
            self.text.write('\n')
        self.current_tag = None
    
    def handle_data(self, data):
        if self.current_tag not in self.skip_tags:
            self.text.write(data)
    
    def get_text(self):
        return self.text.getvalue()

def extract_text_from_html(html_path: Path) -> Optional[str]:
    try:
        with open(html_path, 'r', encoding='utf-8', errors='ignore') as f:
            html_content = f.read()
        parser = HTMLTextExtractor()
        parser.feed(html_content)
        text = parser.get_text()
        lines = [line.strip() for line in text.split('\n')]
        text = '\n'.join(line for line in lines if line)
        return text if text.strip() else None
    except Exception as e:
        print(f"Failed to extract HTML {html_path.name}: {e}")
        return None

def extract_text(file_path: Path) -> Optional[str]:
    suffix = file_path.suffix.lower()
    if suffix == '.pdf':
        return extract_text_from_pdf(file_path)
    elif suffix in ['.htm', '.html']:
        return extract_text_from_html(file_path)
    return None

In [ ]:
# Find all documents

def find_documents(data_dir: Path) -> List[dict]:
    docs = []
    data_dir = Path(data_dir)
    
    for year_dir in sorted(data_dir.iterdir()):
        if not year_dir.is_dir():
            continue
        try:
            year = int(year_dir.name)
        except ValueError:
            continue
        
        for quarter_dir in year_dir.iterdir():
            if not quarter_dir.is_dir():
                continue
            quarter = quarter_dir.name.upper()
            if quarter not in ["Q1", "Q2", "Q3", "Q4"]:
                continue
            
            for pattern in ["*.pdf", "*.htm", "*.html"]:
                for doc in quarter_dir.glob(pattern):
                    docs.append({
                        "path": doc,
                        "year": year,
                        "quarter": quarter,
                        "contract_id": doc.stem,
                    })
    
    print(f"Found {len(docs)} documents")
    return docs

# Count documents
docs = find_documents(DATA_DIR)

In [ ]:
# Load embedding model on GPU
print(f"Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME, device='cuda')
print(f"Model loaded! Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
# Main processing loop - generates embeddings and saves to compressed JSONL

import time
import gzip

data_path = Path(DATA_DIR)
processed = 0
total_chunks = 0
start_time = time.time()

# Check for existing progress (resume capability)
existing_contracts = set()
output_path = Path(OUTPUT_FILE)
is_gzip = OUTPUT_FILE.endswith('.gz')

if output_path.exists():
    opener = gzip.open if is_gzip else open
    with opener(output_path, 'rt') as f:
        for line in f:
            data = json.loads(line)
            existing_contracts.add(data['contract_id'])
    print(f"Resuming: {len(existing_contracts)} contracts already processed")

# Filter out already processed
docs_to_process = [d for d in docs if d['contract_id'] not in existing_contracts]
print(f"Processing {len(docs_to_process)} remaining documents...")

# Open file for appending (gzip or plain)
opener = gzip.open if is_gzip else open
mode = 'at' if output_path.exists() else 'wt'

with opener(OUTPUT_FILE, mode) as out_file:
    batch_texts = []
    batch_metadata = []
    
    for doc_info in tqdm(docs_to_process, desc="Processing"):
        text = extract_text(doc_info["path"])
        if not text:
            continue
        
        chunks = chunk_text(text, max_tokens=CHUNK_SIZE)
        relative_path = str(doc_info["path"].relative_to(data_path))
        file_type = doc_info["path"].suffix.lower().lstrip('.')
        
        for chunk in chunks:
            batch_texts.append(chunk.text)
            batch_metadata.append({
                "contract_id": doc_info["contract_id"],
                "year": doc_info["year"],
                "quarter": doc_info["quarter"],
                "section": chunk.section,
                "chunk_index": chunk.index,
                "text": chunk.text,
                "file_path": relative_path,
                "file_type": file_type,
            })
            
            # Process batch when full
            if len(batch_texts) >= BATCH_SIZE:
                embeddings = model.encode(batch_texts, normalize_embeddings=True, show_progress_bar=False)
                
                for emb, meta in zip(embeddings, batch_metadata):
                    record = {**meta, "embedding": emb.tolist()}
                    out_file.write(json.dumps(record) + '\n')
                
                total_chunks += len(batch_texts)
                batch_texts = []
                batch_metadata = []
        
        processed += 1
        
        # Progress update every 100 docs
        if processed % 100 == 0:
            elapsed = time.time() - start_time
            rate = processed / elapsed * 60
            remaining = (len(docs_to_process) - processed) / rate
            print(f"\n{processed}/{len(docs_to_process)} docs | {rate:.0f} docs/min | ~{remaining:.1f} min remaining")
    
    # Process remaining batch
    if batch_texts:
        embeddings = model.encode(batch_texts, normalize_embeddings=True, show_progress_bar=False)
        for emb, meta in zip(embeddings, batch_metadata):
            record = {**meta, "embedding": emb.tolist()}
            out_file.write(json.dumps(record) + '\n')
        total_chunks += len(batch_texts)

elapsed = time.time() - start_time
print(f"\n\nDone! Processed {processed} documents, {total_chunks} chunks")
print(f"Total time: {elapsed/60:.1f} minutes ({processed/elapsed*60:.0f} docs/min)")
print(f"Output saved to: {OUTPUT_FILE}")

In [ ]:
# Verify output file
import os
file_size = os.path.getsize(OUTPUT_FILE) / (1024 * 1024 * 1024)
print(f"Output file size: {file_size:.2f} GB")

# Count records
opener = gzip.open if OUTPUT_FILE.endswith('.gz') else open
with opener(OUTPUT_FILE, 'rt') as f:
    count = sum(1 for _ in f)
print(f"Total records: {count}")

## Next Steps

1. Download `contract_embeddings.jsonl.gz` from your Google Drive (~2GB)
2. Run the local loader script:

```bash
# Start Qdrant
docker run -d -p 6333:6333 qdrant/qdrant

# Load embeddings into Qdrant (handles .gz automatically)
python scripts/load_embeddings.py --input contract_embeddings.jsonl.gz
```

**Expected GPU speed:** ~200-400 docs/minute (vs 7 docs/min on CPU)  
**Total time estimate:** ~2-3 hours for 40,000 documents